In [1]:
# =========================
# CELL 1 — IMPORTS + SAFETY
# =========================

import cv2
import numpy as np
import time
from IPython.display import display
import ipywidgets as widgets

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

car.throttle = 0.0
car.steering = 0.0

print("Libraries loaded. Car stopped.")

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Libraries loaded. Car stopped.


In [2]:
# =========================
# CELL 2 — CAMERA SETUP
# =========================

camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

time.sleep(2)

image_widget = widgets.Image(format='jpeg', width=448, height=448)
display(image_widget)

def get_frame():
    frame = camera.value

    if frame is None:
        time.sleep(0.5)
        frame = camera.value

    return frame

def show_frame(frame):
    if frame is not None:
        image_widget.value = bgr8_to_jpeg(frame)

frame = get_frame()
show_frame(frame)

print("Camera ready.")
print("Frame is None:", frame is None)

if frame is not None:
    print("Shape:", frame.shape)
    print("Mean brightness:", round(frame.mean(), 2))
    print("Min:", frame.min())
    print("Max:", frame.max())

Image(value=b'', format='jpeg', height='448', width='448')

Camera ready.
Frame is None: False
Shape: (224, 224, 3)
Mean brightness: 122.84
Min: 5
Max: 255


In [3]:
# =========================
# CELL 3 — PARKING CLASSIFIER SETUP
# =========================

CATEGORIES = ['empty_bay', 'occupied_bay', 'background']
CLASSIFIER_MODEL_PATH = 'parking_classifier.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

parking_classifier = torchvision.models.resnet18(pretrained=False)
parking_classifier.fc = torch.nn.Linear(512, len(CATEGORIES))
parking_classifier.load_state_dict(torch.load(CLASSIFIER_MODEL_PATH, map_location=device))
parking_classifier = parking_classifier.to(device)
parking_classifier.eval()

classifier_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

ML_CONFIDENCE_THRESHOLD = 0.70

def classify_parking_scene(frame):
    input_tensor = classifier_transform(frame).unsqueeze(0).to(device)

    with torch.no_grad():
        output = parking_classifier(input_tensor)
        probs = F.softmax(output, dim=1)[0]

    pred_idx = int(torch.argmax(probs).item())
    label = CATEGORIES[pred_idx]
    confidence = float(probs[pred_idx].item())

    probabilities = {
        CATEGORIES[i]: float(probs[i].item())
        for i in range(len(CATEGORIES))
    }

    return {
        "label": label,
        "confidence": confidence,
        "probabilities": probabilities
    }

print("Parking classifier loaded.")
print("Device:", device)
print("Classes:", CATEGORIES)

Parking classifier loaded.
Device: cuda
Classes: ['empty_bay', 'occupied_bay', 'background']


In [4]:
# =========================
# CELL 4 — MOTION CONTROL
# =========================
# Important:
# Your JetRacer moves forward using NEGATIVE throttle.
# Reverse uses POSITIVE throttle.

FORWARD_THROTTLE = -0.22
KICK_THROTTLE = -0.28
REVERSE_THROTTLE = 0.22

STEERING_OFFSET = 0.03
MAX_STEER = 0.70

def stop(duration=0.15):
    car.throttle = 0.0
    car.steering = 0.0
    time.sleep(duration)

def drive(throttle, steering, duration):
    steering = float(np.clip(steering + STEERING_OFFSET, -MAX_STEER, MAX_STEER))
    car.steering = steering
    car.throttle = float(throttle)
    time.sleep(duration)
    stop()

def forward(duration=0.30, steering=0.0):
    car.steering = float(np.clip(steering + STEERING_OFFSET, -MAX_STEER, MAX_STEER))

    # Kick-start to overcome surface resistance
    car.throttle = KICK_THROTTLE
    time.sleep(0.12)

    car.throttle = FORWARD_THROTTLE
    time.sleep(duration)

    stop()

def reverse(duration=0.25, steering=0.0):
    drive(REVERSE_THROTTLE, steering, duration)

def arc_left(duration=0.35, steer=0.55):
    forward(duration=duration, steering=steer)

def arc_right(duration=0.35, steer=0.55):
    forward(duration=duration, steering=-steer)

def reverse_left(duration=0.30, steer=0.55):
    reverse(duration=duration, steering=steer)

def reverse_right(duration=0.30, steer=0.55):
    reverse(duration=duration, steering=-steer)

print("Motion primitives ready.")

Motion primitives ready.


In [5]:
# =========================
# CELL 5 — CAMERA DIAGNOSTIC
# =========================

frame = get_frame()
show_frame(frame)

if frame is None:
    print("ERROR: camera frame is None")
else:
    print("Camera working")
    print("Shape:", frame.shape)
    print("Mean brightness:", round(frame.mean(), 2))
    print("Min:", frame.min())
    print("Max:", frame.max())

Camera working
Shape: (224, 224, 3)
Mean brightness: 129.65
Min: 5
Max: 255


In [6]:
# =========================
# CELL 6 — ROBUST WHITE MASK v6.8
# =========================
# Goal:
#   Keep clean side-tape detection.
#   Improve horizontal back-tape recovery.
#   Remove final far-right blob noise.
#
# This is a targeted refinement of v6.7:
#   1. Smaller horizontal opening kernel to keep fragmented back tape.
#   2. Larger horizontal closing kernel to reconnect broken back tape.
#   3. Final far-right border cleanup after all paths are combined.

def get_white_mask(frame):

    height, width, _ = frame.shape

    roi_start = int(height * 0.32)
    roi_end = int(height * 0.98)

    roi = frame[roi_start:roi_end, :]

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)

    S_MAX = 80
    V_MIN = 155

    raw = np.zeros_like(V)
    raw[(S <= S_MAX) & (V >= V_MIN)] = 255

    roi_h, roi_w = raw.shape

    # -------------------------------------------------
    # PATH A: v6-style side-tape filtering
    # -------------------------------------------------

    filtered = np.zeros_like(raw)

    contours, _ = cv2.findContours(
        raw,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 15:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        long_side = max(w, h)
        short_side = max(min(w, h), 1)
        aspect = long_side / short_side

        # Reject upper glare/wall fragments.
        if y < int(roi_h * 0.18) and area > 700:
            continue

        # Reject huge filled blobs.
        if area > 3500:
            continue

        # Reject far-edge blob noise.
        near_right_edge = x > int(roi_w * 0.82)
        near_left_edge = (x + w) < int(roi_w * 0.08)

        if near_right_edge and area > 50:
            continue

        if near_left_edge and area > 50:
            continue

        # Reject compact chunky blobs.
        blob_like = area > 120 and aspect < 1.7
        if blob_like:
            continue

        # Keep tape-like structures.
        keep_line_like = long_side >= 18 and aspect >= 1.6
        keep_medium_tape_piece = area >= 80 and long_side >= 12 and aspect >= 1.35

        if keep_line_like or keep_medium_tape_piece:
            cv2.drawContours(filtered, [cnt], -1, 255, -1)

    # Reconnect side tape lightly.
    kernel_close = np.ones((3, 3), np.uint8)
    filtered = cv2.morphologyEx(filtered, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # Remove tiny leftovers.
    kernel_open = np.ones((2, 2), np.uint8)
    filtered = cv2.morphologyEx(filtered, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # -------------------------------------------------
    # PATH B: horizontal back-tape recovery
    # -------------------------------------------------
    # The horizontal line is visible in raw HSV, but fragmented.
    # Use a smaller opening kernel so fragments survive,
    # then a longer close kernel to reconnect gaps.

    horizontal_only = np.zeros_like(raw)

    # Slightly wider y-band than v6.7.
    y_band_start = int(roi_h * 0.04)
    y_band_end = int(roi_h * 0.42)

    # Keep central x-band only. This removes far-right false blob risk.
    x_band_start = int(roi_w * 0.12)
    x_band_end = int(roi_w * 0.80)

    band = raw[y_band_start:y_band_end, x_band_start:x_band_end]

    # Smaller opening preserves fragmented back tape.
    horizontal_open_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (6, 1))
    h_mask = cv2.morphologyEx(band, cv2.MORPH_OPEN, horizontal_open_kernel, iterations=1)

    # Longer close reconnects the horizontal fragments.
    horizontal_close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 1))
    h_mask = cv2.morphologyEx(h_mask, cv2.MORPH_CLOSE, horizontal_close_kernel, iterations=1)

    contours_h, _ = cv2.findContours(
        h_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    h_filtered = np.zeros_like(h_mask)

    for cnt in contours_h:
        area = cv2.contourArea(cnt)

        if area < 5:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        long_side = max(w, h)
        short_side = max(min(w, h), 1)
        aspect = long_side / short_side

        # Back tape should be thin, horizontal, and not too large.
        is_back_tape_fragment = (
            w >= 8 and
            h <= 7 and
            aspect >= 1.6 and
            area <= 600
        )

        if is_back_tape_fragment:
            cv2.drawContours(h_filtered, [cnt], -1, 255, -1)

    horizontal_only[
        y_band_start:y_band_end,
        x_band_start:x_band_end
    ] = h_filtered

    # -------------------------------------------------
    # Combine paths
    # -------------------------------------------------

    combined = cv2.bitwise_or(filtered, horizontal_only)

    # -------------------------------------------------
    # Final cleanup: remove far-right blob after combination
    # -------------------------------------------------
    # This is important because the remaining blob can survive through either path.
    # We remove large components close to the right edge unless they are long, thin lines.

    final = np.zeros_like(combined)

    contours_final, _ = cv2.findContours(
        combined,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_final:
        area = cv2.contourArea(cnt)

        if area < 10:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        long_side = max(w, h)
        short_side = max(min(w, h), 1)
        aspect = long_side / short_side

        near_far_right = x > int(roi_w * 0.82)
        near_far_left = (x + w) < int(roi_w * 0.06)

        # Remove border blobs.
        # Real tape can be near the side, but it should be line-like.
        if near_far_right and area > 35 and aspect < 2.5:
            continue

        if near_far_left and area > 35 and aspect < 2.5:
            continue

        # Remove compact non-tape blobs anywhere.
        if area > 90 and aspect < 1.45:
            continue

        cv2.drawContours(final, [cnt], -1, 255, -1)

    full_mask = np.zeros((height, width), dtype=np.uint8)
    full_mask[roi_start:roi_end, :] = final

    return full_mask

In [7]:
# CELL 7 — ROBUST WHITE MASK v6.8 FROM parkingbay_aligning_v8
# Purpose:
# Restore the known-good white tape mask from the working v8 notebook.
# This detects actual white tape pixels instead of drawing artificial Hough blobs.

def get_white_mask(frame):

    height, width, _ = frame.shape

    roi_start = int(height * 0.32)
    roi_end = int(height * 0.98)

    roi = frame[roi_start:roi_end, :]

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)

    S_MAX = 80
    V_MIN = 155

    raw = np.zeros_like(V)
    raw[(S <= S_MAX) & (V >= V_MIN)] = 255

    roi_h, roi_w = raw.shape

    # PATH A: side-tape filtering
    filtered = np.zeros_like(raw)

    contours, _ = cv2.findContours(
        raw,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 15:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        long_side = max(w, h)
        short_side = max(min(w, h), 1)
        aspect = long_side / short_side

        # Reject upper glare/wall fragments.
        if y < int(roi_h * 0.18) and area > 700:
            continue

        # Reject huge filled blobs.
        if area > 3500:
            continue

        # Reject far-edge blob noise.
        near_right_edge = x > int(roi_w * 0.82)
        near_left_edge = (x + w) < int(roi_w * 0.08)

        if near_right_edge and area > 50:
            continue

        if near_left_edge and area > 50:
            continue

        # Reject compact chunky blobs.
        blob_like = area > 120 and aspect < 1.7
        if blob_like:
            continue

        # Keep tape-like structures.
        keep_line_like = long_side >= 18 and aspect >= 1.6
        keep_medium_tape_piece = area >= 80 and long_side >= 12 and aspect >= 1.35

        if keep_line_like or keep_medium_tape_piece:
            cv2.drawContours(filtered, [cnt], -1, 255, -1)

    # Reconnect side tape lightly.
    kernel_close = np.ones((3, 3), np.uint8)
    filtered = cv2.morphologyEx(filtered, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # Remove tiny leftovers.
    kernel_open = np.ones((2, 2), np.uint8)
    filtered = cv2.morphologyEx(filtered, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # PATH B: horizontal back-tape recovery
    horizontal_only = np.zeros_like(raw)

    y_band_start = int(roi_h * 0.04)
    y_band_end = int(roi_h * 0.42)

    x_band_start = int(roi_w * 0.12)
    x_band_end = int(roi_w * 0.80)

    band = raw[y_band_start:y_band_end, x_band_start:x_band_end]

    horizontal_open_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (6, 1))
    h_mask = cv2.morphologyEx(band, cv2.MORPH_OPEN, horizontal_open_kernel, iterations=1)

    horizontal_close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 1))
    h_mask = cv2.morphologyEx(h_mask, cv2.MORPH_CLOSE, horizontal_close_kernel, iterations=1)

    contours_h, _ = cv2.findContours(
        h_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    h_filtered = np.zeros_like(h_mask)

    for cnt in contours_h:
        area = cv2.contourArea(cnt)

        if area < 5:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        long_side = max(w, h)
        short_side = max(min(w, h), 1)
        aspect = long_side / short_side

        is_back_tape_fragment = (
            w >= 8 and
            h <= 7 and
            aspect >= 1.6 and
            area <= 600
        )

        if is_back_tape_fragment:
            cv2.drawContours(h_filtered, [cnt], -1, 255, -1)

    horizontal_only[
        y_band_start:y_band_end,
        x_band_start:x_band_end
    ] = h_filtered

    # Combine side and horizontal paths.
    combined = cv2.bitwise_or(filtered, horizontal_only)

    # Final cleanup.
    final = np.zeros_like(combined)

    contours_final, _ = cv2.findContours(
        combined,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_final:
        area = cv2.contourArea(cnt)

        if area < 10:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        long_side = max(w, h)
        short_side = max(min(w, h), 1)
        aspect = long_side / short_side

        near_far_right = x > int(roi_w * 0.82)
        near_far_left = (x + w) < int(roi_w * 0.06)

        if near_far_right and area > 35 and aspect < 2.5:
            continue

        if near_far_left and area > 35 and aspect < 2.5:
            continue

        if area > 90 and aspect < 1.45:
            continue

        cv2.drawContours(final, [cnt], -1, 255, -1)

    full_mask = np.zeros((height, width), dtype=np.uint8)
    full_mask[roi_start:roi_end, :] = final

    return full_mask

print("Cell 6 restored from v8.")

Cell 6 restored from v8.


In [8]:
# CELL 8 — U-SHAPE BAY DETECTOR v3 FROM parkingbay_aligning_v8
# Purpose:
# Detect bay geometry mainly from the two side tapes.
# The horizontal back tape is useful but not required.

def line_angle_degrees(x1, y1, x2, y2):
    return np.degrees(np.arctan2(y2 - y1, x2 - x1))


def detect_parking_bay(frame):
    height, width, _ = frame.shape
    image_center_x = width // 2

    mask = get_white_mask(frame)
    debug = frame.copy()

    roi_start = int(height * 0.32)
    roi_end = int(height * 0.96)
    roi = mask[roi_start:roi_end, :]

    edges = cv2.Canny(roi, 50, 150)

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=14,
        minLineLength=18,
        maxLineGap=14
    )

    side_lines = []
    rejected_lines = []

    if lines is not None:
        for raw in lines:
            x1, y1, x2, y2 = raw[0]

            y1_full = y1 + roi_start
            y2_full = y2 + roi_start

            dx = x2 - x1
            dy = y2_full - y1_full
            length = float(np.sqrt(dx * dx + dy * dy))

            if length < 18:
                rejected_lines.append((x1, y1_full, x2, y2_full, "short"))
                continue

            angle = line_angle_degrees(x1, y1_full, x2, y2_full)

            # Ignore near-horizontal lines for main side-line geometry.
            if abs(angle) < 28:
                rejected_lines.append((x1, y1_full, x2, y2_full, "horizontal"))
                continue

            if abs(angle) > 160:
                rejected_lines.append((x1, y1_full, x2, y2_full, "bad_angle"))
                continue

            if min(x1, x2) < 6 or max(x1, x2) > width - 6:
                rejected_lines.append((x1, y1_full, x2, y2_full, "border"))
                continue

            bottom_y = max(y1_full, y2_full)
            if bottom_y < int(height * 0.48):
                rejected_lines.append((x1, y1_full, x2, y2_full, "too_high"))
                continue

            mid_x = (x1 + x2) // 2
            if mid_x < 20 or mid_x > width - 20:
                rejected_lines.append((x1, y1_full, x2, y2_full, "outer_zone"))
                continue

            if y1_full > y2_full:
                bx, by = x1, y1_full
                tx, ty = x2, y2_full
            else:
                bx, by = x2, y2_full
                tx, ty = x1, y1_full

            vertical_span = abs(by - ty)
            if vertical_span < 24:
                rejected_lines.append((x1, y1_full, x2, y2_full, "low_vertical_span"))
                continue

            side_lines.append({
                "bottom": (int(bx), int(by)),
                "top": (int(tx), int(ty)),
                "mid_x": int(mid_x),
                "angle": float(angle),
                "length": float(length),
                "vertical_span": int(vertical_span),
                "raw": (int(x1), int(y1_full), int(x2), int(y2_full))
            })

            cv2.line(debug, (x1, y1_full), (x2, y2_full), (0, 255, 0), 2)
            cv2.circle(debug, (int(bx), int(by)), 4, (0, 255, 255), -1)

    mode = "NONE"
    entrance_center = None
    back_center = None
    centerline_angle = None
    entrance_width = None
    bay_detected = False

    best_pair = None
    best_score = 999999

    if len(side_lines) >= 2:
        for i in range(len(side_lines)):
            for j in range(i + 1, len(side_lines)):
                a = side_lines[i]
                b = side_lines[j]

                left, right = (a, b) if a["bottom"][0] < b["bottom"][0] else (b, a)

                lb = left["bottom"]
                rb = right["bottom"]
                lt = left["top"]
                rt = right["top"]

                width_bottom = rb[0] - lb[0]
                width_top = rt[0] - lt[0]

                if width_bottom < 45 or width_bottom > 170:
                    continue

                if width_top < 25 or width_top > 170:
                    continue

                pair_mid_x = (lb[0] + rb[0]) // 2
                if abs(pair_mid_x - image_center_x) > 65:
                    continue

                bottom_y_diff = abs(lb[1] - rb[1])
                if bottom_y_diff > 45:
                    continue

                top_y_diff = abs(lt[1] - rt[1])
                if top_y_diff > 45:
                    continue

                entrance_x = (lb[0] + rb[0]) // 2
                entrance_y = (lb[1] + rb[1]) // 2

                back_x = (lt[0] + rt[0]) // 2
                back_y = (lt[1] + rt[1]) // 2

                dx_center = entrance_x - back_x
                dy_center = entrance_y - back_y

                if abs(dy_center) < 8:
                    continue

                center_angle = np.degrees(np.arctan2(dx_center, dy_center))

                if abs(center_angle) > 50:
                    continue

                left_dist = abs(image_center_x - lb[0])
                right_dist = abs(rb[0] - image_center_x)
                symmetry_error = abs(left_dist - right_dist)

                length_diff = abs(left["length"] - right["length"])

                expected_width = 115

                score = (
                    abs(entrance_x - image_center_x) * 0.30 +
                    abs(width_bottom - expected_width) * 0.18 +
                    abs(width_bottom - width_top) * 0.12 +
                    abs(center_angle) * 0.75 +
                    symmetry_error * 0.15 +
                    bottom_y_diff * 0.10 +
                    top_y_diff * 0.08 +
                    length_diff * 0.05 -
                    (left["length"] + right["length"]) * 0.04
                )

                if score < best_score:
                    best_score = score
                    best_pair = {
                        "left": left,
                        "right": right,
                        "entrance_center": (int(entrance_x), int(entrance_y)),
                        "back_center": (int(back_x), int(back_y)),
                        "centerline_angle": float(center_angle),
                        "entrance_width": int(width_bottom),
                        "score": float(score),
                        "width_top": int(width_top),
                        "bottom_y_diff": int(bottom_y_diff),
                        "top_y_diff": int(top_y_diff),
                        "symmetry_error": int(symmetry_error)
                    }

    if best_pair is not None:
        mode = "FULL_U_BAY"
        bay_detected = True

        entrance_center = best_pair["entrance_center"]
        back_center = best_pair["back_center"]
        centerline_angle = best_pair["centerline_angle"]
        entrance_width = best_pair["entrance_width"]

        left = best_pair["left"]
        right = best_pair["right"]

        cv2.line(debug, left["bottom"], left["top"], (0, 255, 0), 4)
        cv2.line(debug, right["bottom"], right["top"], (0, 255, 0), 4)

        cv2.circle(debug, left["bottom"], 7, (0, 255, 0), -1)
        cv2.circle(debug, right["bottom"], 7, (0, 255, 0), -1)
        cv2.line(debug, left["bottom"], right["bottom"], (255, 255, 0), 3)

        cv2.circle(debug, left["top"], 5, (255, 255, 0), -1)
        cv2.circle(debug, right["top"], 5, (255, 255, 0), -1)
        cv2.line(debug, left["top"], right["top"], (255, 255, 255), 2)

        cv2.circle(debug, entrance_center, 8, (255, 0, 0), -1)
        cv2.circle(debug, back_center, 7, (255, 0, 255), -1)
        cv2.line(debug, back_center, entrance_center, (255, 0, 255), 3)

    elif len(side_lines) == 1:
        mode = "PARTIAL"
        bay_detected = True

        only_line = side_lines[0]
        cv2.line(debug, only_line["bottom"], only_line["top"], (0, 165, 255), 3)
        cv2.circle(debug, only_line["bottom"], 8, (0, 165, 255), -1)

    else:
        mode = "NONE"
        bay_detected = False

    cv2.line(debug, (image_center_x, 0), (image_center_x, height), (0, 0, 255), 2)

    cv2.putText(
        debug,
        f"mode:{mode} side_lines:{len(side_lines)}",
        (10, 24),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (0, 255, 255),
        1
    )

    cv2.putText(
        debug,
        f"width:{entrance_width} angle:{None if centerline_angle is None else round(centerline_angle, 1)}",
        (10, 47),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (0, 255, 255),
        1
    )

    if entrance_center is not None:
        x_error = entrance_center[0] - image_center_x
        cv2.putText(
            debug,
            f"xerr:{x_error}",
            (10, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 255, 255),
            1
        )

    diagnostics = {
        "mode": mode,
        "bay_detected": bay_detected,
        "entrance_center": entrance_center,
        "back_center": back_center,
        "centerline_angle": centerline_angle,
        "entrance_width": entrance_width,
        "confidence": 1.0 if mode == "FULL_U_BAY" else (0.35 if mode == "PARTIAL" else 0.0),
        "side_line_count": len(side_lines),
        "rejected_line_count": len(rejected_lines),
        "score": None if best_pair is None else best_pair["score"],
        "width_top": None if best_pair is None else best_pair["width_top"],
        "bottom_y_diff": None if best_pair is None else best_pair["bottom_y_diff"],
        "top_y_diff": None if best_pair is None else best_pair["top_y_diff"],
        "symmetry_error": None if best_pair is None else best_pair["symmetry_error"]
    }

    return diagnostics, debug

print("Cell 7 restored from v8.")

Cell 7 restored from v8.


In [9]:
# =========================
# CELL 9 — DETECTOR DIAGNOSTIC
# =========================

frame = get_frame()
diagnostics, debug = detect_parking_bay(frame)

show_frame(debug)

print("=== Bay Detection Diagnostics ===")
for k, v in diagnostics.items():
    print(k, ":", v)

if diagnostics["entrance_center"] is not None:
    x_error = diagnostics["entrance_center"][0] - (camera.width // 2)
    print("x_error:", x_error)

=== Bay Detection Diagnostics ===
mode : NONE
bay_detected : False
entrance_center : None
back_center : None
centerline_angle : None
entrance_width : None
confidence : 0.0
side_line_count : 0
rejected_line_count : 0
score : None
width_top : None
bottom_y_diff : None
top_y_diff : None
symmetry_error : None


In [10]:
# =========================
# CELL 10 — STABLE POSE SCAN
# =========================

def scan_bay_pose(num_frames=7, delay=0.06):
    full = []
    partial = 0
    debug_last = None

    for _ in range(num_frames):
        frame = get_frame()

        if frame is None:
            time.sleep(delay)
            continue

        diagnostics, debug = detect_parking_bay(frame)
        debug_last = debug

        if diagnostics["mode"] == "FULL_U_BAY":
            full.append(diagnostics)
        elif diagnostics["mode"] == "PARTIAL":
            partial += 1

        time.sleep(delay)

    if len(full) > 0:
        xs = [d["entrance_center"][0] for d in full]
        ys = [d["entrance_center"][1] for d in full]
        angles = [d["centerline_angle"] for d in full if d["centerline_angle"] is not None]
        widths = [d["entrance_width"] for d in full if d["entrance_width"] is not None]

        result = {
            "mode": "FULL_U_BAY",
            "confidence": len(full) / num_frames,
            "entrance_center": (int(np.mean(xs)), int(np.mean(ys))),
            "centerline_angle": float(np.mean(angles)) if len(angles) > 0 else None,
            "entrance_width": int(np.mean(widths)) if len(widths) > 0 else None,
            "debug": debug_last
        }

        return result

    if partial > 0:
        return {
            "mode": "PARTIAL",
            "confidence": partial / num_frames,
            "entrance_center": None,
            "centerline_angle": None,
            "entrance_width": None,
            "debug": debug_last
        }

    return {
        "mode": "NONE",
        "confidence": 0.0,
        "entrance_center": None,
        "centerline_angle": None,
        "entrance_width": None,
        "debug": debug_last
    }

print("Stable pose scan ready.")

Stable pose scan ready.


In [11]:
# =========================
# CELL 11 — STATIC POSE TEST
# =========================

pose = scan_bay_pose(num_frames=8)

if pose["debug"] is not None:
    show_frame(pose["debug"])

print("Mode:", pose["mode"])
print("Confidence:", round(pose["confidence"], 2))
print("Entrance center:", pose["entrance_center"])
print("Centerline angle:", pose["centerline_angle"])
print("Entrance width:", pose["entrance_width"])

if pose["entrance_center"] is not None:
    print("x_error:", pose["entrance_center"][0] - (camera.width // 2))
    print("y_position:", pose["entrance_center"][1])

Mode: NONE
Confidence: 0.0
Entrance center: None
Centerline angle: None
Entrance width: None


In [43]:
# =========================
# CELL 12 — CONTROLLER HELPERS v17
# =========================
# Goal:
#   Stop at the entrance, not inside the parking bay.
#
# Main fix:
#   Earlier versions kept correcting after y was already very large.
#   That caused the car to enter the bay.
#
# New rule:
#   BUMPER_STOP_Y is earlier.
#   MAX_SAFE_Y creates a hard stop if the bay is already too close.

IMAGE_CENTER_X = camera.width // 2

STOP_X_TOL = 12
STOP_ANGLE_TOL = 8.0

CORRECT_ANGLE_TOL = 7.0
CORRECT_X_TOL = 14

LARGE_X_ERROR = 14
ANGLE_OK_FOR_LATERAL_PRIORITY = 12.0

# Earlier stop target.
# v16 allowed y=169, which was too deep.
BUMPER_STOP_Y = 148

# Hard safety stop. If detected entrance is this low in the image,
# do not correct further. Stop because the bumper is likely at/inside entrance.
MAX_SAFE_Y = 158

NEAR_TARGET_Y = 140

MIN_CONFIDENCE = 0.50


def recover_from_partial_left():
    print("RECOVER: partial bay → left recovery")
    reverse(duration=0.08, steering=0.0)
    arc_left(duration=0.10, steer=0.30)


def recover_from_partial_right():
    print("RECOVER: partial bay → right recovery")
    reverse(duration=0.08, steering=0.0)
    arc_right(duration=0.10, steer=0.30)


def search_for_bay_right():
    print("SEARCH: small right sweep")
    arc_right(duration=0.12, steer=0.45)


def search_for_bay_left():
    print("SEARCH: small left sweep")
    arc_left(duration=0.12, steer=0.45)


def extended_search_for_bay_right():
    print("SEARCH: extended right sweep")
    reverse(duration=0.08, steering=0.0)
    arc_right(duration=0.16, steer=0.55)


def extended_search_for_bay_left():
    print("SEARCH: extended left sweep")
    reverse(duration=0.08, steering=0.0)
    arc_left(duration=0.16, steer=0.55)


def correct_heading(angle):
    """
    Heading correction.
    Only safe before the robot reaches the entrance.
    v17.4: more aggressive for steep angles.
    """
    print("CORRECT HEADING | angle:", round(angle, 2))

    if abs(angle) > 40:
        duration = 0.18
        steer = 0.70
    elif abs(angle) > 30:
        duration = 0.15
        steer = 0.68
    elif abs(angle) > 25:
        duration = 0.13
        steer = 0.65
    elif abs(angle) > 18:
        duration = 0.11
        steer = 0.58
    elif abs(angle) > 12:
        duration = 0.09
        steer = 0.50
    elif abs(angle) > 7:
        duration = 0.07
        steer = 0.42
    else:
        duration = 0.05
        steer = 0.34

    reverse(duration=0.06, steering=0.0)

    if angle > 0:
        arc_right(duration=duration, steer=steer)
    else:
        arc_left(duration=duration, steer=steer)


def correct_lateral(x_error):
    """
    Lateral correction with corrected sign.
    """
    print("CORRECT LATERAL | x_error:", x_error)

    if abs(x_error) > 35:
        duration = 0.13
        steer = 0.52
    elif abs(x_error) > 24:
        duration = 0.10
        steer = 0.46
    else:
        duration = 0.07
        steer = 0.38

    if x_error > 0:
        arc_right(duration=duration, steer=steer)
    else:
        arc_left(duration=duration, steer=steer)


def approach_with_visual_servo(x_error, angle):
    """
    Short forward creep only.
    """
    steering = -0.007 * x_error - 0.005 * angle
    steering = float(np.clip(steering, -0.22, 0.22))

    print("APPROACH | steering:", round(steering, 3))
    forward(duration=0.075, steering=steering)


def final_offset_creep(duration=0.12):
    """
    Very small final creep only.
    Previous 0.22 could push the car too far in.
    """
    print("FINAL OFFSET CREEP | duration:", duration)
    forward(duration=duration, steering=0.0)


def guided_sweep_left(strength="small"):
    if strength == "large":
        print("SEARCH: large left sweep")
        arc_left(duration=0.11, steer=0.45)
    elif strength == "medium":
        print("SEARCH: medium left sweep")
        arc_left(duration=0.09, steer=0.35)
    else:
        print("SEARCH: small left sweep")
        arc_left(duration=0.07, steer=0.25)


def guided_sweep_right(strength="small"):
    if strength == "large":
        print("SEARCH: large right sweep")
        arc_right(duration=0.11, steer=0.45)
    elif strength == "medium":
        print("SEARCH: medium right sweep")
        arc_right(duration=0.09, steer=0.35)
    else:
        print("SEARCH: small right sweep")
        arc_right(duration=0.07, steer=0.25)


        
def align_toward_bay(direction):
    """
    Three-phase maneuver:
    1. Steer aggressively toward the bay
    2. Steer back to straighten up
    3. Reverse to create distance for CV detection
    """
def align_toward_bay(direction):
    if direction == "left":
        print("ALIGN: aggressive left, straighten right, then reverse")
        arc_left(duration=1.7, steer=0.70)
        arc_right(duration=1.4, steer=0.55)
    else:
        print("ALIGN: aggressive right, straighten left, then reverse")
        arc_right(duration=1.7, steer=0.70)
        arc_left(duration=1.4, steer=0.55)
    
    print("ALIGN: reversing for CV pickup")
    reverse(duration=1.2, steering=0.0)
    
print("Controller helpers v17 ready.")
print("BUMPER_STOP_Y:", BUMPER_STOP_Y)
print("MAX_SAFE_Y:", MAX_SAFE_Y)
print("NEAR_TARGET_Y:", NEAR_TARGET_Y)

Controller helpers v17 ready.
BUMPER_STOP_Y: 148
MAX_SAFE_Y: 158
NEAR_TARGET_Y: 140


In [13]:
# =========================
# CELL 13 — MAIN PARKING ALIGNMENT CONTROLLER v17.3
# =========================
# Goal:
#   Stop at the parking bay entrance.
#
# v17.3 change:
#   When classifier sees empty_bay but CV has no geometry,
#   classify left/right halves of frame to determine which
#   direction the bay is in, and steer that way.

def classify_direction(frame):
    """
    Classify left and right halves of frame separately.
    Returns 'left', 'right', or 'center' based on which
    half scores higher for empty_bay.
    """
    height, width, _ = frame.shape
    mid = width // 2

    left_half = frame[:, :mid, :]
    right_half = frame[:, mid:, :]

    # Resize halves back to 224x224 for the classifier
    left_resized = cv2.resize(left_half, (224, 224))
    right_resized = cv2.resize(right_half, (224, 224))

    cl_left = classify_parking_scene(left_resized)
    cl_right = classify_parking_scene(right_resized)

    left_score = cl_left["probabilities"].get("empty_bay", 0.0)
    right_score = cl_right["probabilities"].get("empty_bay", 0.0)

    print("DIRECTION CHECK | left_score:", round(left_score, 2),
          "| right_score:", round(right_score, 2))

    diff = abs(left_score - right_score)

    if diff < 0.15:
        return "center"
    elif left_score > right_score:
        return "left"
    else:
        return "right"


def run_parking_alignment(max_steps=200):
    print("Parking alignment controller v17.3 started")

    last_good_pose = None
    last_good_step = None
    last_good_debug = None

    LOST_VIEW_FINAL_CREEP_DURATION = 0.12

    none_count = 0
    partial_count = 0
    same_direction_count = 0
    last_search_direction = None
    has_aligned = False


    try:
        for step in range(1, max_steps + 1):
            print("\n--- ALIGNMENT LOOP", step, "---")
            stop()

            pose = scan_bay_pose(num_frames=7)

            if pose["debug"] is not None and pose["mode"] != "NONE":
                show_frame(pose["debug"])

            # -------------------------------------------------
            # CLASSIFIER — runs every iteration
            # -------------------------------------------------
            frame = get_frame()
            cl_label = "unknown"
            cl_conf = 0.0
            cl_probs = {}

            if frame is not None:
                cl = classify_parking_scene(frame)
                cl_label = cl["label"]
                cl_conf = cl["confidence"]
                cl_probs = cl["probabilities"]
                print("CLASSIFIER:", cl_label,
                      "| conf:", round(cl_conf, 2),
                      "| probs:", {k: round(v, 2) for k, v in cl_probs.items()})

            # Override PARTIAL/FULL_U_BAY if classifier says not a real bay
            if pose["mode"] in ("PARTIAL", "FULL_U_BAY"):
                if cl_label == "background" and cl_conf >= ML_CONFIDENCE_THRESHOLD:
                    print("CLASSIFIER OVERRIDE: background, treating as NONE")
                    pose["mode"] = "NONE"
                    pose["confidence"] = 0.0
                    pose["entrance_center"] = None
                    pose["centerline_angle"] = None
                    pose["entrance_width"] = None

                elif cl_label == "occupied_bay" and cl_conf >= ML_CONFIDENCE_THRESHOLD:
                    print("CLASSIFIER OVERRIDE: occupied bay, treating as NONE")
                    pose["mode"] = "NONE"
                    pose["confidence"] = 0.0
                    pose["entrance_center"] = None
                    pose["centerline_angle"] = None
                    pose["entrance_width"] = None

            print("Mode:", pose["mode"],
                  "| Conf:", round(pose["confidence"], 2),
                  "| Center:", pose["entrance_center"],
                  "| Angle:", pose["centerline_angle"],
                  "| Width:", pose["entrance_width"])

            # -------------------------------------------------
            # NONE
            # -------------------------------------------------
            if pose["mode"] == "NONE":
                none_count += 1

                if last_good_pose is not None:
                    last_cx, last_cy = last_good_pose["entrance_center"]
                    last_angle = last_good_pose["centerline_angle"]
                    last_x_error = last_cx - IMAGE_CENTER_X

                    recently_seen = (step - last_good_step) <= 2
                    close_to_target = last_cy >= NEAR_TARGET_Y
                    reasonably_aligned = (
                        abs(last_x_error) <= STOP_X_TOL and
                        abs(last_angle) <= STOP_ANGLE_TOL
                    )

                    if recently_seen and close_to_target and reasonably_aligned:
                        print("SUCCESS: bay lost after close aligned approach")
                        print("Last good y:", last_cy,
                              "| last x_error:", last_x_error,
                              "| last angle:", round(last_angle, 2))

                        if last_good_debug is not None:
                            show_frame(last_good_debug)

                        final_offset_creep(duration=LOST_VIEW_FINAL_CREEP_DURATION)
                        stop()
                        return True

                if none_count >= 200 and last_good_pose is None:
                    print("STOPPED: no reliable bay detected after extended search")
                    stop()
                    return False

                # Classifier sees a bay but CV can't find lines —
                # use split-frame classification to determine direction
                if cl_label == "empty_bay" and cl_conf >= ML_CONFIDENCE_THRESHOLD and frame is not None:
                    direction = classify_direction(frame)

                    frame_dir = get_frame()
                    if frame_dir is not None:
                        h_d, w_d, _ = frame_dir.shape
                        mid_d = w_d // 2
                        cl_l = classify_parking_scene(cv2.resize(frame_dir[:, :mid_d, :], (224, 224)))
                        cl_r = classify_parking_scene(cv2.resize(frame_dir[:, mid_d:, :], (224, 224)))
                        left_score = cl_l["probabilities"]["empty_bay"]
                        right_score = cl_r["probabilities"]["empty_bay"]
                        dominant_score = max(left_score, right_score)
                    else:
                        left_score = 0.5
                        right_score = 0.5
                        dominant_score = 0.5

                    if dominant_score > 0.85:
                        strength = "large"
                    elif dominant_score > 0.5:
                        strength = "medium"
                    else:
                        strength = "small"

                    # Both sides similar = bay is roughly centered, creep forward
                    both_high = left_score > 0.3 and right_score > 0.3
                    scores_close = abs(left_score - right_score) < 0.35
                    
                    # Bay is far to one side — align toward it then straighten
                    # Bay is far to one side — align toward it then straighten
                    far_left = left_score > 0.85 and right_score < 0.10
                    far_right = right_score > 0.85 and left_score < 0.10

                    if not has_aligned and (far_left or far_right):
                        if far_left:
                            print("INITIAL ALIGN: bay far left")
                            align_toward_bay("left")
                        else:
                            print("INITIAL ALIGN: bay far right")
                            align_toward_bay("right")
                        has_aligned = True
                        same_direction_count = 0
                        continue

                    if far_left and same_direction_count > 2:
                        print("ALIGN: bay far left, repositioning")
                        align_toward_bay("left")
                        same_direction_count = 0
                        continue

                    if far_right and same_direction_count > 2:
                        print("ALIGN: bay far right, repositioning")
                        align_toward_bay("right")
                        same_direction_count = 0
                        continue
                    
                    if both_high and scores_close:
                        print("SEARCH: bay centered, creeping forward | L:",
                              round(left_score, 2), "R:", round(right_score, 2))
                        forward(duration=0.18, steering=0.0)
                        same_direction_count = 0
                    elif direction == "left":
                        if last_search_direction == "left":
                            same_direction_count += 1
                        else:
                            same_direction_count = 1
                            last_search_direction = "left"

                        if same_direction_count > 4 and same_direction_count % 3 == 0:
                            print("SEARCH: stuck sweeping LEFT, arc-forward to reposition")
                            forward(duration=0.15, steering=0.0)
                            arc_left(duration=0.13, steer=0.50)
                        else:
                            print("SEARCH: classifier guided sweep LEFT |", strength)
                            guided_sweep_left(strength)
                    elif direction == "right":
                        if last_search_direction == "right":
                            same_direction_count += 1
                        else:
                            same_direction_count = 1
                            last_search_direction = "right"

                        if same_direction_count > 4 and same_direction_count % 3 == 0:
                            print("SEARCH: stuck sweeping RIGHT, arc-forward to reposition")
                            forward(duration=0.15, steering=0.0)
                            arc_right(duration=0.13, steer=0.50)
                        else:
                            print("SEARCH: classifier guided sweep RIGHT |", strength)
                            guided_sweep_right(strength)
                    else:
                        print("SEARCH: classifier sees bay, creeping forward")
                        forward(duration=0.09, steering=0.0)
                        same_direction_count = 0
                    continue

                # No classifier signal — fall back to blind search pattern
                phase = none_count % 16

                if phase in [1, 2, 3, 4]:
                    search_for_bay_right()
                elif phase in [5, 6, 7, 8]:
                    search_for_bay_left()
                elif phase in [9, 10, 11, 12]:
                    extended_search_for_bay_right()
                else:
                    extended_search_for_bay_left()

                continue

            none_count = 0

            # -------------------------------------------------
            # PARTIAL — use direction check to recover smartly
            # -------------------------------------------------
            if pose["mode"] == "PARTIAL":
                partial_count += 1

                if cl_label == "empty_bay" and cl_conf >= ML_CONFIDENCE_THRESHOLD and frame is not None:
                    direction = classify_direction(frame)

                    frame_dir = get_frame()
                    if frame_dir is not None:
                        h_d, w_d, _ = frame_dir.shape
                        mid_d = w_d // 2
                        cl_l = classify_parking_scene(cv2.resize(frame_dir[:, :mid_d, :], (224, 224)))
                        cl_r = classify_parking_scene(cv2.resize(frame_dir[:, mid_d:, :], (224, 224)))
                        dominant_score = max(cl_l["probabilities"]["empty_bay"],
                                             cl_r["probabilities"]["empty_bay"])
                    else:
                        dominant_score = 0.5

                    if dominant_score > 0.85:
                        strength = "large"
                    elif dominant_score > 0.5:
                        strength = "medium"
                    else:
                        strength = "small"

                    if direction == "left":
                        print("RECOVER: classifier guided LEFT |", strength)
                        guided_sweep_left(strength)
                    elif direction == "right":
                        print("RECOVER: classifier guided RIGHT |", strength)
                        guided_sweep_right(strength)
                    else:
                        print("RECOVER: classifier says center, creeping forward")
                        forward(duration=0.15, steering=0.0)
                else:
                    if partial_count % 2 == 1:
                        recover_from_partial_left()
                    else:
                        recover_from_partial_right()

                continue

            partial_count = 0

            # -------------------------------------------------
            # FULL_U_BAY
            # -------------------------------------------------
            if pose["entrance_center"] is None or pose["centerline_angle"] is None:
                print("Rejected FULL_U_BAY: missing geometry")
                recover_from_partial_right()
                continue

            cx, cy = pose["entrance_center"]
            x_error = cx - IMAGE_CENTER_X
            angle = pose["centerline_angle"]

            print("x_error:", x_error, "| angle:", round(angle, 2), "| y:", cy)

            weak_conf = pose["confidence"] < MIN_CONFIDENCE
            angle_plausible = abs(angle) <= 12
            centre_plausible = abs(x_error) <= 45

            if weak_conf:
                if not angle_plausible and abs(x_error) > 25:
                    print("REJECTED weak geometry:",
                          "conf =", round(pose["confidence"], 2),
                          "| x_error =", x_error,
                          "| angle =", round(angle, 2))
                    continue

                if not centre_plausible:
                    print("REJECTED weak geometry: centre too extreme",
                          "| x_error =", x_error)
                    continue

                print("Weak confidence but usable geometry")

            if last_good_pose is not None:
                last_cx, last_cy = last_good_pose["entrance_center"]
                last_angle = last_good_pose["centerline_angle"]
                last_x_error = last_cx - IMAGE_CENTER_X

                jump_x = abs(x_error - last_x_error)
                jump_angle = abs(angle - last_angle)

                if weak_conf and not angle_plausible and (jump_x > 30 or jump_angle > 15):
                    print("REJECTED unstable geometry jump:",
                          "jump_x =", jump_x,
                          "| jump_angle =", round(jump_angle, 2))
                    continue

            last_good_pose = pose
            last_good_step = step
            last_good_debug = pose["debug"]

            stop_aligned = (
                abs(x_error) <= STOP_X_TOL and
                abs(angle) <= STOP_ANGLE_TOL
            )

            at_bumper_stop = cy >= BUMPER_STOP_Y
            hard_stop = cy >= MAX_SAFE_Y
            near_target = cy >= NEAR_TARGET_Y

            correction_needed_angle = abs(angle) > CORRECT_ANGLE_TOL
            correction_needed_x = abs(x_error) >= CORRECT_X_TOL

            lateral_priority = (
                abs(x_error) >= LARGE_X_ERROR and
                abs(angle) <= ANGLE_OK_FOR_LATERAL_PRIORITY
            )

            # -------------------------------------------------
            # HARD STOP
            # -------------------------------------------------
            if hard_stop:
                print("SUCCESS: hard stop at entrance safety limit")
                print("Final x_error:", x_error,
                      "| final angle:", round(angle, 2),
                      "| final y:", cy)
                stop()
                return True

            # -------------------------------------------------
            # NORMAL ENTRANCE STOP
            # -------------------------------------------------
            if at_bumper_stop:
                if abs(x_error) <= STOP_X_TOL and abs(angle) <= STOP_ANGLE_TOL:
                    print("SUCCESS: stopped at parking bay entrance")
                    print("Final x_error:", x_error,
                          "| final angle:", round(angle, 2),
                          "| final y:", cy)
                    stop()
                    return True

                print("SUCCESS: entrance reached, stopping before entering bay")
                print("Final x_error:", x_error,
                      "| final angle:", round(angle, 2),
                      "| final y:", cy)
                stop()
                return True

            # -------------------------------------------------
            # NEAR TARGET
            # -------------------------------------------------
            if near_target:
                if lateral_priority or correction_needed_x:
                    print("NEAR TARGET: x_error high, small lateral correction")
                    correct_lateral(x_error)
                    continue

                if correction_needed_angle:
                    print("NEAR TARGET: angle high, small heading correction")
                    correct_heading(angle)
                    continue

                approach_with_visual_servo(x_error, angle)
                continue

            # -------------------------------------------------
            # FAR FROM TARGET
            # -------------------------------------------------
            if lateral_priority:
                print("LATERAL PRIORITY: angle manageable, x_error high")
                correct_lateral(x_error)
                continue

            if correction_needed_angle:
                correct_heading(angle)
                continue

            if correction_needed_x:
                correct_lateral(x_error)
                continue

            approach_with_visual_servo(x_error, angle)

        print("FAILED: safety step limit reached")
        stop()
        return False

    except KeyboardInterrupt:
        print("\nINTERRUPT: emergency stop")
        car.throttle = 0.0
        car.steering = 0.0
        return False


print("Main controller v17.3 ready.")

Main controller v17.3 ready.


In [14]:
# =========================
# CELL 14 — RUN PARKING ALIGNMENT
# =========================

success = run_parking_alignment(max_steps=300)

if success:
    print("FINAL RESULT: SUCCESS")
else:
    print("FINAL RESULT: FAILED")

Parking alignment controller v17.3 started

--- ALIGNMENT LOOP 1 ---
CLASSIFIER: background | conf: 0.51 | probs: {'empty_bay': 0.49, 'occupied_bay': 0.0, 'background': 0.51}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 2 ---
CLASSIFIER: background | conf: 0.88 | probs: {'empty_bay': 0.11, 'occupied_bay': 0.0, 'background': 0.88}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 3 ---
CLASSIFIER: background | conf: 0.89 | probs: {'empty_bay': 0.11, 'occupied_bay': 0.0, 'background': 0.89}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 4 ---
CLASSIFIER: background | conf: 0.62 | probs: {'empty_bay': 0.37, 'occupied_bay': 0.0, 'background': 0.62}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 5 ---
CLASSIFIER: empty_bay | conf: 0.65 

In [15]:
# =========================
# CELL 15 — EMERGENCY STOP
# =========================
"""
stop()
car.throttle = 0.0
car.steering = 0.0

print("Emergency stop executed.")
"""

'\nstop()\ncar.throttle = 0.0\ncar.steering = 0.0\n\nprint("Emergency stop executed.")\n'

In [35]:
# =========================
# CELL 16 — BAY ENTRY HELPERS
# =========================
# Once the car is at the bay entrance, drive in using
# the side tapes for centering and detect the back wall
# to know when to stop.

def detect_side_tapes(frame):
    """
    Look for left and right side tapes while inside the bay.
    Tapes will be near the left and right edges of the frame.
    """
    height, width, _ = frame.shape
    mask = get_white_mask(frame)
    
    roi_top = int(height * 0.50)
    roi_bottom = int(height * 0.95)
    roi = mask[roi_top:roi_bottom, :]
    
    roi_h, roi_w = roi.shape
    
    # Look for tape in the left third of the image
    left_region = roi[:, :roi_w // 3]
    # Look for tape in the right third
    right_region = roi[:, 2 * roi_w // 3:]
    
    left_x = None
    right_x = None
    
    left_cols = np.where(left_region.sum(axis=0) > 0)[0]
    if len(left_cols) > 3:
        left_x = int(np.mean(left_cols))
    
    right_cols = np.where(right_region.sum(axis=0) > 0)[0]
    if len(right_cols) > 3:
        right_x = int(np.mean(right_cols) + 2 * roi_w // 3)
    
    return left_x, right_x

def detect_back_tape(frame):
    """
    Detect the horizontal back tape of the bay.
    Returns the y position (in full image coords) of the back tape,
    or None if not detected.
    """
    height, width, _ = frame.shape
    mask = get_white_mask(frame)
    
    # Wider ROI to catch the back tape as it moves down the image
    roi_top = int(height * 0.20)
    roi_bottom = int(height * 0.85)
    roi = mask[roi_top:roi_bottom, :]
    
    roi_h, roi_w = roi.shape
    
    best_y = None
    best_width = 0
    
    for y in range(roi_h):
        row = roi[y, :]
        white_cols = np.where(row > 0)[0]
        
        if len(white_cols) < 15:
            continue
        
        span = white_cols[-1] - white_cols[0]
        
        if span > roi_w * 0.30 and span > best_width:
            best_width = span
            best_y = y + roi_top
    
    return best_y


def compute_lane_steering(left_x, right_x, frame_width):
    """
    Compute steering to stay centered between the two side tapes.
    """
    center_x = frame_width // 2
    
    if left_x is not None and right_x is not None:
        lane_center = (left_x + right_x) // 2
        error = lane_center - center_x
        steering = -0.01 * error  # negative because positive steering = physical left
        return float(np.clip(steering, -0.25, 0.25))
    
    elif left_x is not None:
        # Only see left tape — steer away from it (right)
        error = left_x - int(frame_width * 0.3)
        steering = -0.008 * error
        return float(np.clip(steering, -0.20, 0.20))
    
    elif right_x is not None:
        # Only see right tape — steer away from it (left)
        error = right_x - int(frame_width * 0.7)
        steering = -0.008 * error
        return float(np.clip(steering, -0.20, 0.20))
    
    return 0.0


print("Bay entry helpers ready.")

Bay entry helpers ready.


In [39]:
# =========================
# CELL 17 — BAY ENTRY CONTROLLER
# =========================
# Drive into the bay using side tapes for lane-keeping
# and back tape detection to know when to stop.

BACK_TAPE_STOP_Y = 155  # much closer — tape needs to be low in image
BAY_ENTRY_THROTTLE = -0.18
BAY_ENTRY_MAX_STEPS = 80
BAY_ENTRY_IGNORE_STEPS = 5  # ignore back tape for first N steps while entering
BACK_TAPE_SEEN_MIN = 5


def run_bay_entry(max_steps=BAY_ENTRY_MAX_STEPS):
    back_tape_seen_count = 0
    back_tape_lost_count = 0
    print("Bay entry controller started")
    print("Driving into bay...")
    
    no_tape_count = 0
    
    try:
        for step in range(1, max_steps + 1):
            print("\n--- BAY ENTRY LOOP", step, "---")
            
            frame = get_frame()
            if frame is None:
                time.sleep(0.1)
                continue
            
            show_frame(frame)
            
            height, width, _ = frame.shape
            
            # Check for back tape (ignore early steps while still entering)
            # Check for back tape
            back_y = detect_back_tape(frame)
            
            if back_y is not None:
                back_tape_seen_count += 1
                back_tape_lost_count = 0
                print("BACK TAPE detected at y:", back_y, "| seen:", back_tape_seen_count)
            else:
                if back_tape_seen_count >= BACK_TAPE_SEEN_MIN:
                    back_tape_lost_count += 1
                    print("BACK TAPE lost | lost count:", back_tape_lost_count)
                    
                    if back_tape_lost_count >= 3:
                        print("SUCCESS: back tape disappeared after being visible — reached end of bay")
                        stop()
                        return True
            
            # Get side tapes for lane keeping
            left_x, right_x = detect_side_tapes(frame)
            print("Side tapes | left:", left_x, "| right:", right_x, "| back:", back_y)
            
            if left_x is None and right_x is None:
                no_tape_count += 1
                print("No side tapes visible, count:", no_tape_count)
                
                if no_tape_count >= 15:
                    print("STOPPED: lost all tape references")
                    stop()
                    return False
                
                # Drive straight slowly
                steering = 0.0
            else:
                no_tape_count = 0
                steering = compute_lane_steering(left_x, right_x, width)
            
            print("ENTRY | steering:", round(steering, 3))
            
            # Drive forward slowly with lane-keeping steering
            car.steering = float(np.clip(steering + STEERING_OFFSET, -MAX_STEER, MAX_STEER))
            car.throttle = KICK_THROTTLE
            time.sleep(0.10)
            car.throttle = BAY_ENTRY_THROTTLE
            time.sleep(0.20)
            car.throttle = 0.0
            time.sleep(0.05)
        
        print("FAILED: max steps reached without finding back tape")
        stop()
        return False
    
    except KeyboardInterrupt:
        print("\nINTERRUPT: emergency stop")
        car.throttle = 0.0
        car.steering = 0.0
        return False


print("Bay entry controller ready.")
print("BACK_TAPE_STOP_Y:", BACK_TAPE_STOP_Y)

Bay entry controller ready.
BACK_TAPE_STOP_Y: 155


In [44]:
# =========================
# CELL 18 — RUN FULL PARKING SEQUENCE
# =========================

# Phase 1: Find and align with bay entrance
print("=" * 50)
print("PHASE 1: ALIGNMENT")
print("=" * 50)

alignment_success = run_parking_alignment(max_steps=300)

if alignment_success:
    print("\nAlignment succeeded. Starting bay entry in 1 second...")
    time.sleep(1.0)
    
    # Phase 2: Drive into the bay
    print("=" * 50)
    print("PHASE 2: BAY ENTRY")
    print("=" * 50)
    
    entry_success = run_bay_entry(max_steps=200)
    
    if entry_success:
        print("\n" + "=" * 50)
        print("PARKING COMPLETE!")
        print("=" * 50)
    else:
        print("\n" + "=" * 50)
        print("BAY ENTRY FAILED")
        print("=" * 50)
else:
    print("\nAlignment failed. Cannot enter bay.")
    print("PARKING FAILED")

PHASE 1: ALIGNMENT
Parking alignment controller v17.3 started

--- ALIGNMENT LOOP 1 ---
CLASSIFIER: background | conf: 0.74 | probs: {'empty_bay': 0.26, 'occupied_bay': 0.0, 'background': 0.74}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 2 ---
CLASSIFIER: background | conf: 0.73 | probs: {'empty_bay': 0.26, 'occupied_bay': 0.0, 'background': 0.73}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 3 ---
CLASSIFIER: background | conf: 0.52 | probs: {'empty_bay': 0.48, 'occupied_bay': 0.01, 'background': 0.52}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 4 ---
CLASSIFIER: empty_bay | conf: 0.63 | probs: {'empty_bay': 0.63, 'occupied_bay': 0.01, 'background': 0.37}
Mode: NONE | Conf: 0.0 | Center: None | Angle: None | Width: None
SEARCH: small right sweep

--- ALIGNMENT LOOP 5 ---
CLASSIFIER: bac

In [19]:
# =========================
# CELL 19 — EMERGENCY STOP
# =========================

stop()
car.throttle = 0.0
car.steering = 0.0

print("Emergency stop executed.")

Emergency stop executed.
